In [ ]:

import torch
from torch.utils.data import TensorDataset, DataLoader
data_dict = torch.load('PyTorch/MLP/data/tcga_processed_tensors.pt')
    
# 拆包提取
train_loader = data_dict['train_loader']
valid_loader = data_dict['valid_loader']
test_loader = data_dict['test_loader']
input_dim = data_dict['input_dim']
y_train = data_dict['y_train']
num_classes = data_dict['num_classes']

/var/folders/0g/1st6ryj54n1bshvyq3yt1pzm0000gn/T/ipykernel_71261/1622867860.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data_dict = torch.load('/Users/zengjian/PyTor

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# 1. 模型构建
print("model build begin")
class TCGA_CancerMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Linear -> BatchNorm -> ReLU -> Dropout
        self.fc1 = nn.Linear(input_dim, 1024) #全连接层
        self.bn1 = nn.BatchNorm1d(1024) # 批归一化，加速收敛
        self.relu1 = nn.ReLU() # 激活函数
        self.drop1 = nn.Dropout(0.5)    # Dropout 防止过拟合
        
        # 隐藏层 1 到隐藏层 2
        self.fc2 = nn.Linear(1024, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)
        
        # 输出层 (映射到cancer type)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.drop1(self.relu1(self.bn1(self.fc1(x))))
        x = self.drop2(self.relu2(self.bn2(self.fc2(x))))
        x = self.fc3(x) # CrossEntropyLoss 自带 Softmax，这里不需要加
        return x

# 2. 初始化模型、损失函数与优化器
# 定义输入维度和输出类别

model = TCGA_CancerMLP(input_dim, num_classes)

# 计算类别权重，对抗样本不均匀
# y_train 是之前划分出来的训练集标签 (numpy 数组格式)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# 定义加权交叉熵损失，带类别权重
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# 定义优化器 (Adam 优化器在组学数据上表现非常稳健)，带L2正则化
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# 3. 训练循环，开始训练

num_epochs = 20
l1_lambda = 1e-5  # L1 惩罚系数，根据实际 Loss 规模微调
best_val_acc = 0.0

print(f"model train begin...\n")

for epoch in range(num_epochs):
    # --- 训练阶段 ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_x, batch_y in train_loader:
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(batch_x)
        
        # 构建总损失 = 加权分类损失 + L1 正则化
        classification_loss = criterion(outputs, batch_y)
        
        # 只对第一层的权重 (fc1) 施加 L1 惩罚，迫使无用的基因权重变为 0
        l1_penalty = torch.norm(model.fc1.weight, p=1)
        
        # 总损失计算
        loss = classification_loss + l1_lambda * l1_penalty
        
        # 反向传播与权重更新
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # 统计训练准确率
        _, predicted = torch.max(outputs.data, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()

    train_acc = 100 * correct_train / total_train
    
    # 4. 验证阶段 
    model.eval()  # 关闭 Dropout 和 BatchNorm的更新
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # 验证阶段不计算梯度，节省显存
        for batch_x, batch_y in valid_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total_val += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()
            
    val_acc = 100 * correct_val / total_val
    
    # 打印每个 Epoch 的日志
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss/len(valid_loader):.4f}, Val Acc: {val_acc:.2f}%")
    
    # 5. 模型保存，保存准确率最高的模型
    # 如果当前验证集准确率是历史上最好的，就保存模型权重
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_tcga_mlp.pth')

print(f"\n 训练完成！最高验证集准确率: {best_val_acc:.2f}%")
print("已将最佳模型权重保存为 'best_tcga_mlp.pth'")

model build begin
model train begin...

Epoch [1/20] Train Loss: 1.7813, Train Acc: 79.76% | Val Loss: 0.4034, Val Acc: 89.94%
Epoch [2/20] Train Loss: 1.0965, Train Acc: 90.32% | Val Loss: 0.3327, Val Acc: 89.34%
Epoch [3/20] Train Loss: 1.0734, Train Acc: 90.62% | Val Loss: 0.3349, Val Acc: 91.27%
Epoch [4/20] Train Loss: 0.9861, Train Acc: 92.50% | Val Loss: 0.3662, Val Acc: 91.27%
Epoch [5/20] Train Loss: 1.1555, Train Acc: 91.99% | Val Loss: 0.3705, Val Acc: 89.82%
Epoch [6/20] Train Loss: 1.1380, Train Acc: 92.09% | Val Loss: 0.3509, Val Acc: 90.42%
Epoch [7/20] Train Loss: 1.2973, Train Acc: 92.18% | Val Loss: 0.3726, Val Acc: 91.33%
Epoch [8/20] Train Loss: 1.1386, Train Acc: 92.82% | Val Loss: 0.3916, Val Acc: 89.94%
Epoch [9/20] Train Loss: 1.2148, Train Acc: 92.95% | Val Loss: 0.3526, Val Acc: 91.93%
Epoch [10/20] Train Loss: 1.2025, Train Acc: 93.39% | Val Loss: 0.3986, Val Acc: 91.14%
Epoch [11/20] Train Loss: 1.2325, Train Acc: 93.26% | Val Loss: 0.3241, Val Acc: 92.41%
E